# Task 2: Distributed Analytics with PySpark
Compute the average annual temperature for each weather station using PySpark distributed aggregation.

In [1]:
import os
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import year, col, avg, round
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

os.environ["JAVA_HOME"] = "/opt/homebrew/Cellar/openjdk@21/21.0.10/libexec/openjdk.jdk/Contents/Home" # Replace with your own path

In [2]:
spark = SparkSession.builder.appName("NOAA Pipeline - Task 2").getOrCreate()

myschema = StructType([
    StructField("STATION", IntegerType(), False),
    StructField("DATE", DateType(), True),
    StructField("LATITUDE", DoubleType(), True),
    StructField("LONGITUDE", DoubleType(), True),
    StructField("ELEVATION", DoubleType(), True),
    StructField("NAME", StringType(), True),
    StructField("TEMP", DoubleType(), True),
    StructField("TEMP_ATTRIBUTES", IntegerType(), True),
    StructField("DEWP", DoubleType(), True),
    StructField("DEWP_ATTRIBUTES", IntegerType(), True),
    StructField("SLP", DoubleType(), True),
    StructField("SLP_ATTRIBUTES", IntegerType(), True),
    StructField("STP", DoubleType(), True),
    StructField("STP_ATTRIBUTES", IntegerType(), True),
    StructField("VISIB", DoubleType(), True),
    StructField("VISIB_ATTRIBUTES", IntegerType(), True),
    StructField("WDSP", DoubleType(), True),
    StructField("WDSP_ATTRIBUTES", IntegerType(), True),
    StructField("MXSPD", DoubleType(), True),
    StructField("GUST", DoubleType(), True),
    StructField("MAX", DoubleType(), True),
    StructField("MAX_ATTRIBUTES", StringType(), True),
    StructField("MIN", DoubleType(), True),
    StructField("MIN_ATTRIBUTES", StringType(), True),
    StructField("PRCP", DoubleType(), True),
    StructField("PRCP_ATTRIBUTES", StringType(), True),
    StructField("SNDP", DoubleType(), True),
    StructField("FRSHTT", StringType(), True),
])

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 00:53:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load Data from Shared Folder

In [3]:
shared_path = str(Path.cwd().parent / "shared")

df_combine = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(myschema)
    .load(shared_path))

print(f"Loaded {df_combine.count()} records")

Loaded 48404 records


## Filter Invalid / Missing Temperature Values
NOAA uses 9999.9 as a sentinel for missing data. We also remove nulls.

In [4]:
df_clean = df_combine.filter(
    (col("TEMP").isNotNull()) &
    (col("TEMP") != 9999.9)
)

removed = df_combine.count() - df_clean.count()
print(f"Removed {removed} invalid/missing temperature records")

Removed 0 invalid/missing temperature records


## Compute Average Annual Temperature per Station
Extract station identifier and year from DATE, then use distributed groupBy aggregation.

In [7]:
df_avg_temp = (
    df_clean
    .select(
        col("STATION"),
        year(col("DATE")).alias("YEAR"),
        col("TEMP")
    )
    .groupBy("STATION", "YEAR")
    .agg(round(avg("TEMP"), 2).alias("AVG_TEMP"))
    .orderBy("STATION", "YEAR")
)

df_avg_temp.show(10, truncate=False)

+----------+----+--------+
|STATION   |YEAR|AVG_TEMP|
+----------+----+--------+
|1001099999|2022|30.36   |
|1001099999|2023|32.31   |
|1001099999|2024|33.42   |
|1001099999|2025|34.24   |
|1001499999|2022|49.5    |
|1001499999|2023|48.69   |
|1001499999|2024|49.81   |
|1001499999|2025|50.24   |
|1002099999|2023|19.0    |
|1002099999|2024|26.15   |
+----------+----+--------+
only showing top 10 rows


## Save Output

In [6]:
output_path = str(Path.cwd().parent / "task2_output")

df_avg_temp.coalesce(1).write.csv(path=output_path, mode="overwrite", header=True)
print("=" * 20)
print("Task 2 output saved successfully")
print("=" * 20)

Task 2 output saved successfully
